### Implement cross-validation with different folds by designing custom k-fold and stratified split mechanisms, ensuring balanced class representation, and analyzing variance in model performance across iterations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("heart_disease.csv")

print("Dataset shape:", df.shape)
print(df.head())
print("Columns:", df.columns.tolist())

target_col = "target"

if target_col not in df.columns:
    raise ValueError(
        f"Target '{target_col}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

y = pd.Series(pd.factorize(y)[0], name=target_col)

print("Features:", X.shape)
print("Target distribution:")
print(y.value_counts())


Dataset shape: (25, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
Features: (25, 13)
Target distribution:
target
0    16
1     9
Name: count, dtype: int64


In [2]:
def custom_kfold(n_samples, k=5, shuffle=True, random_state=42):
    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    indices = np.arange(n_samples)

    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, k)

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate(
            [folds[j] for j in range(k) if j != i]
        )

        yield train_idx, val_idx


for fold_no, (train_idx, val_idx) in enumerate(
    custom_kfold(len(X), k=5), start=1
):
    print(
        f"Fold {fold_no}: "
        f"Train = {len(train_idx)}, "
        f"Validation = {len(val_idx)}"
    )


Fold 1: Train = 20, Validation = 5
Fold 2: Train = 20, Validation = 5
Fold 3: Train = 20, Validation = 5
Fold 4: Train = 20, Validation = 5
Fold 5: Train = 20, Validation = 5


In [3]:
def custom_stratified_kfold(
    y, k=5, shuffle=True, random_state=42
):
    y = np.asarray(y)
    n_samples = len(y)

    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    classes, counts = np.unique(y, return_counts=True)

    if counts.min() < k:
        raise ValueError(
            "Each class must have at least k samples "
            "for every validation fold to contain that class."
        )

    rng = np.random.default_rng(random_state)
    fold_parts = [[] for _ in range(k)]

    for cls in classes:
        cls_idx = np.where(y == cls)[0]

        if shuffle:
            rng.shuffle(cls_idx)

        class_folds = np.array_split(cls_idx, k)

        for fold_no, part in enumerate(class_folds):
            fold_parts[fold_no].extend(part.tolist())

    all_indices = np.arange(n_samples)

    for fold_no in range(k):
        val_idx = np.array(fold_parts[fold_no], dtype=int)

        if shuffle:
            rng.shuffle(val_idx)

        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        yield train_idx, val_idx


for fold_no, (train_idx, val_idx) in enumerate(
    custom_stratified_kfold(y, k=5), start=1
):
    print(f"\nFold {fold_no}")
    print("Validation size:", len(val_idx))
    print("Class counts:")
    print(pd.Series(y.iloc[val_idx]).value_counts().sort_index())



Fold 1
Validation size: 6
Class counts:
target
0    4
1    2
Name: count, dtype: int64

Fold 2
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 3
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 4
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 5
Validation size: 4
Class counts:
target
0    3
1    1
Name: count, dtype: int64


In [4]:
models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=5, random_state=42)
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
}

In [6]:
print("Target class distribution:")
print(y.value_counts())

print("\nMinimum samples in any class:")
print(y.value_counts().min())

Target class distribution:
target
0    16
1     9
Name: count, dtype: int64

Minimum samples in any class:
9


In [7]:
def custom_stratified_kfold(
    y, k=5, shuffle=True, random_state=42
):
    y = np.asarray(y)
    n_samples = len(y)

    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    classes, counts = np.unique(y, return_counts=True)

    min_class_count = counts.min()

    if min_class_count < k:
        raise ValueError(
            f"Cannot use {k} stratified folds because the "
            f"smallest class has only {min_class_count} samples. "
            f"Use k <= {min_class_count}."
        )

    rng = np.random.default_rng(random_state)
    fold_parts = [[] for _ in range(k)]

    for cls in classes:
        class_idx = np.where(y == cls)[0]

        if shuffle:
            rng.shuffle(class_idx)

        for fold_no, part in enumerate(np.array_split(class_idx, k)):
            fold_parts[fold_no].extend(part.tolist())

    all_indices = np.arange(n_samples)

    for fold_no in range(k):
        val_idx = np.array(fold_parts[fold_no], dtype=int)

        if shuffle:
            rng.shuffle(val_idx)

        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False

        yield all_indices[train_mask], val_idx


In [10]:
min_class_count = int(y.value_counts().min())

k_stratified = min(10, min_class_count)

print("Smallest class count:", min_class_count)
print("Stratified folds:", k_stratified)

results_10_kfold = evaluate_custom_cv(
    X, y, models, k=10, method="kfold"
)

results_stratified = evaluate_custom_cv(
    X, y, models, k=k_stratified, method="stratified"
)

all_results = pd.concat(
    [
        results_5_kfold,
        results_5_stratified,
        results_10_kfold,
        results_stratified
    ],
    ignore_index=True
)

print(all_results.head(10))


Smallest class count: 9
Stratified folds: 9
  Method  K                Model  Fold  Train_Size  Validation_Size  Accuracy  \
0  kfold  5  Logistic Regression     1          20                5       0.6   
1  kfold  5  Logistic Regression     2          20                5       1.0   
2  kfold  5  Logistic Regression     3          20                5       1.0   
3  kfold  5  Logistic Regression     4          20                5       1.0   
4  kfold  5  Logistic Regression     5          20                5       0.6   
5  kfold  5        Decision Tree     1          20                5       1.0   
6  kfold  5        Decision Tree     2          20                5       1.0   
7  kfold  5        Decision Tree     3          20                5       0.6   
8  kfold  5        Decision Tree     4          20                5       1.0   
9  kfold  5        Decision Tree     5          20                5       0.6   

   Precision  Recall        F1  
0       0.80     0.6  0.566667 

In [14]:
results_5_kfold = evaluate_custom_cv(
    X, y, models, k=5, method="kfold"
)

results_5_stratified = evaluate_custom_cv(
    X, y, models, k=5, method="stratified"
)

results_10_kfold = evaluate_custom_cv(
    X, y, models, k=10, method="kfold"
)

min_class_count = int(y.value_counts().min())
k_stratified = min(10, min_class_count)

results_stratified = evaluate_custom_cv(
    X, y, models,
    k=k_stratified,
    method="stratified"
)

all_results = pd.concat(
    [
        results_5_kfold,
        results_5_stratified,
        results_10_kfold,
        results_stratified
    ],
    ignore_index=True
)

print("Stratified fold count used:", k_stratified)
display(all_results.head(10))


Stratified fold count used: 9


,Method,K,Model,Fold,Train_Size,Validation_Size,Accuracy,Precision,Recall,F1
0,kfold,5,Logistic Regression,1,20,5,0.6,0.80,0.6,0.566667
1,kfold,5,Logistic Regression,2,20,5,1.0,1.00,1.0,1.000000
2,kfold,5,Logistic Regression,3,20,5,1.0,1.00,1.0,1.000000
3,kfold,5,Logistic Regression,4,20,5,1.0,1.00,1.0,1.000000
4,kfold,5,Logistic Regression,5,20,5,0.6,0.60,0.6,0.600000
5,kfold,5,Decision Tree,1,20,5,1.0,1.00,1.0,1.000000
6,kfold,5,Decision Tree,2,20,5,1.0,1.00,1.0,1.000000
7,kfold,5,Decision Tree,3,20,5,0.6,0.36,0.6,0.450000
8,kfold,5,Decision Tree,4,20,5,1.0,1.00,1.0,1.000000
9,kfold,5,Decision Tree,5,20,5,0.6,0.60,0.6,0.600000


### Document findings on robustness testing by generating a structured technical report, detailing experimental setups, performance trends, failure modes, and recommendations for improving model resilience.

# Technical Report: Robustness Testing of a Heart Disease Prediction Model

## 1. Executive Summary

This report documents a robustness-testing framework for a machine learning model developed to predict heart disease. The objective is to evaluate whether the model produces reliable predictions when faced with changes in input features, unseen data patterns, different training-validation splits, and challenging prediction cases.

The evaluation framework covers adversarial perturbation testing, out-of-distribution (OOD) testing, custom K-Fold and Stratified K-Fold cross-validation, error analysis, and fairness considerations.

The model's final robustness level must be determined from the experimental results. High accuracy alone is insufficient to establish that the model is reliable in real-world healthcare settings.

## 2. Objectives

The primary objectives are:

- Evaluate the stability of predictions under small changes in input features.
- Identify performance degradation on unusual or previously unseen samples.
- Measure variation in performance across cross-validation folds.
- Investigate false-positive and false-negative predictions.
- Examine potential differences in performance across relevant patient subgroups.
- Recommend improvements to enhance model reliability and generalization.

## 3. Experimental Setup

### 3.1 Dataset and preprocessing

The experiments use the `heart_disease.csv` dataset. The dataset contains patient-related features and a target variable indicating heart disease status.

The preprocessing workflow includes:

1. Inspecting dataset dimensions, feature types, and target distribution.
2. Identifying missing values and handling them appropriately.
3. Separating input features from the target variable.
4. Encoding categorical variables where required.
5. Scaling numerical features for models that are sensitive to feature magnitude.
6. Separating training and evaluation data where applicable.

Preprocessing transformations should be fitted only on the training portion of each split to prevent data leakage.

### 3.2 Models evaluated

The planned model comparison includes:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier

The final report should specify the selected model, hyperparameters, and training configuration actually used in the experiments.

### 3.3 Evaluation metrics

The following metrics are used to assess model performance:

| Metric | Purpose |
|---|---|
| Accuracy | Measures overall prediction correctness. |
| Precision | Measures how many predicted positive cases are actually positive. |
| Recall | Measures how many actual positive cases are detected. |
| F1-score | Balances precision and recall. |
| Confusion matrix | Identifies true positives, true negatives, false positives, and false negatives. |
| Standard deviation | Measures variation in performance across validation folds. |
| Prediction confidence | Assesses the model's confidence in individual predictions, where supported. |

For a healthcare application, recall and false-negative analysis are particularly important because missed positive cases may have serious consequences. The appropriate operating threshold requires domain-specific evaluation.

## 4. Robustness Experiment 1: Adversarial Perturbation Testing

### Objective

To investigate whether small, plausible changes in input features cause substantial changes in predictions or prediction confidence.

### Experimental procedure

1. Select representative samples from the test dataset.
2. Generate perturbed copies of these samples.
3. Apply small changes to selected numerical features.
4. Ensure modified values remain within plausible ranges and satisfy known feature constraints.
5. Predict outcomes using the original and perturbed samples.
6. Compare prediction changes, confidence changes, and evaluation metrics.

### Performance analysis

The following measurements should be reported:

- Original test accuracy, precision, recall, and F1-score.
- Corresponding metrics on perturbed test samples.
- Percentage of predictions that change after perturbation.
- Average change in predicted probability, if probability estimates are available.
- Features associated with the largest prediction changes.

### Findings

**Result status:** Requires experimental output.

The model's sensitivity to perturbations cannot be determined without comparing original and modified predictions. A high prediction-flip rate under small, clinically plausible changes may indicate instability.

Perturbations must not create impossible patient profiles. For example, independently modifying related physiological measurements without respecting their relationships can produce unrealistic samples.

## 5. Robustness Experiment 2: Out-of-Distribution Testing

### Objective

To evaluate how well the model behaves on samples that differ from the distribution represented in the training dataset.

### Experimental procedure

1. Inspect the minimum, maximum, mean, and standard deviation of numerical features.
2. Identify unusual values and combinations of features.
3. Construct synthetic edge cases within medically plausible limits.
4. Evaluate the model on these samples without including them in training.
5. Compare predictions and confidence with results from ordinary test samples.
6. Record unsupported inputs, unstable predictions, and potential extrapolation errors.

### Performance analysis

Useful measurements include:

- Accuracy, precision, recall, and F1-score when reliable ground-truth labels are available.
- Distribution differences between training and test samples.
- Confidence changes between in-distribution and OOD samples.
- Percentage of unusual samples flagged by the chosen detection method.
- Performance degradation relative to ordinary test data.

Synthetic samples without verified labels can be used to study prediction behavior, but they cannot establish actual predictive accuracy.

### Findings

**Result status:** Requires experimental output.

A model may produce confident predictions for unfamiliar inputs even when those predictions are incorrect. Therefore, confidence should not be treated as proof of correctness or as a substitute for OOD detection.

## 6. Robustness Experiment 3: Custom Cross-Validation

### Objective

To evaluate whether the model's performance remains consistent across different training and validation subsets.

### Experimental procedure

Two custom splitting strategies are implemented:

**Custom K-Fold:** Divides observations into approximately equal folds, with each fold used once for validation.

**Custom Stratified K-Fold:** Distributes samples from each target class across the folds to preserve class proportions as closely as possible.

The planned comparison includes 5-fold and 10-fold K-Fold, plus stratified cross-validation where the class counts permit it.

During implementation, the smallest target class was found to contain nine samples. Consequently, the custom stratified splitter cannot use ten folds while guaranteeing that every validation fold contains at least one sample from every class. A maximum of nine folds is possible under this requirement.

### Performance analysis

For every model and configuration, calculate:

- Mean accuracy and F1-score across folds.
- Standard deviation and variance of accuracy.
- Standard deviation of F1-score.
- Minimum and maximum fold-level performance.
- Per-fold class counts for stratified validation.

A low standard deviation indicates greater consistency across the tested folds, but does not guarantee strong performance on external datasets.

### Findings

**Result status:** Numerical summary required.

The final report should identify the model with the strongest combination of mean performance and fold-level stability. Results from configurations with different fold counts should be interpreted cautiously because the training-set sizes and validation subsets differ.

## 7. Robustness Experiment 4: Misclassification and Failure-Mode Analysis

### Objective

To understand the types of cases the model predicts incorrectly and identify opportunities for improvement.

### Experimental procedure

1. Generate predictions for the held-out test dataset.
2. Construct a confusion matrix.
3. Extract false-positive and false-negative observations.
4. Compare feature distributions between correctly and incorrectly classified samples.
5. Investigate possible data-quality issues, overlapping classes, and poorly represented feature combinations.
6. Evaluate potential adjustments using validation data rather than repeatedly tuning against the test set.

### Key failure modes

| Failure mode | Potential consequence | Investigation |
|---|---|---|
| False negatives | Heart disease cases may be missed. | Examine recall and missed positive cases. |
| False positives | Additional clinical assessments may be triggered. | Examine precision and false-positive rate. |
| Sparse feature combinations | Predictions may be unreliable for rare profiles. | Review class counts and feature distributions. |
| Overfitting | Training performance may not generalize. | Compare training and validation performance. |
| Poor calibration | Predicted probabilities may not reflect actual event frequencies. | Evaluate calibration curves and Brier score. |
| Data leakage | Evaluation results may appear artificially strong. | Audit preprocessing and data splitting. |

### Findings

**Result status:** Requires confusion-matrix and sample-level analysis.

Each important failure mode should be supported by observed examples or quantitative measurements. Individual prediction explanations may help identify contributing features, but they do not establish causation.

## 8. Fairness and Subgroup Robustness

### Objective

To determine whether predictive performance differs across relevant patient subgroups, provided the dataset contains appropriate attributes and sufficient samples.

### Evaluation procedure

1. Identify relevant demographic or clinical subgroups.
2. Calculate recall, precision, false-positive rate, and false-negative rate for each group.
3. Compare subgroup sample sizes and confidence intervals.
4. Investigate disparities and possible differences in data quality or representation.
5. Evaluate potential mitigation techniques using training data and validation procedures.

Possible approaches include sample reweighting, improved data representation, and carefully evaluated preprocessing changes.

Adversarial debiasing should only be introduced if its assumptions, implementation requirements, and benefits can be evaluated appropriately.

### Findings

**Result status:** Requires subgroup-specific measurements.

A difference in performance does not automatically establish the cause of a disparity. Small subgroup sizes can produce highly uncertain estimates, and fairness improvements must be assessed alongside overall predictive performance.

## 9. Recommendations for Improving Model Resilience

### 9.1 Improve data quality

- Check missing values, duplicate records, inconsistent units, and invalid measurements.
- Investigate class imbalance and underrepresented patient profiles.
- Use clinically plausible feature constraints when generating synthetic samples.

### 9.2 Strengthen validation

- Use stratified validation when class proportions matter.
- Consider repeated stratified cross-validation when class counts permit.
- Keep a separate test set for final evaluation.
- Use external validation data, if available, to assess generalization.
- Prevent preprocessing leakage by fitting transformations inside each training fold.

### 9.3 Improve prediction reliability

- Evaluate probability calibration using appropriate metrics.
- Choose classification thresholds based on validated clinical requirements.
- Assess sensitivity to small input changes.
- Consider OOD detection and flagging uncertain cases for additional review.

### 9.4 Reduce overfitting

- Tune model complexity using training and validation data.
- Compare training and validation scores.
- Evaluate regularization and feature selection.
- Prefer models that achieve a suitable balance between predictive performance and stability.

### 9.5 Maintain monitoring

For any future deployment, monitor data drift, prediction distributions, error rates when labels become available, and subgroup performance. Healthcare use would require appropriate clinical validation, privacy safeguards, and expert oversight.

## 10. Results Summary Template

Complete the table with the values produced by the notebook.

| Experiment | Measurements to report | Current status |
|---|---|---|
| Baseline model | Accuracy, precision, recall, F1-score | Enter measured results |
| Adversarial testing | Prediction-flip rate and metric changes | Pending measurements |
| OOD testing | Distribution shift and confidence changes | Pending measurements |
| Cross-validation | Mean and standard deviation across folds | Calculate from results |
| Failure analysis | False positives and false negatives | Requires sample analysis |
| Fairness analysis | Subgroup metrics and uncertainty | Requires subgroup analysis |

## 11. Conclusion

The robustness-testing framework provides a structured approach to examining the reliability of the Heart Disease Prediction model beyond ordinary test accuracy.

Adversarial perturbation testing evaluates sensitivity to input changes, OOD testing investigates behavior on unfamiliar samples, and custom cross-validation measures the stability of performance across different data splits. Misclassification analysis and subgroup evaluation provide further insight into potential safety and fairness concerns.

The final model should be selected using measured predictive performance, fold-level stability, clinically relevant error rates, and generalization evidence. No model should be considered robust solely because it performs well on one train-test split.